In [1]:
import pandas as pd
import collections

In [2]:
import pandas as pd
import collections

policy_cited_scopus1 = pd.read_csv('/disks/qnap3/users/21-tomokiyo/How_Science_Informs_Policy/bi-gram/2023_Climate Modeling.csv',index_col=0)

In [3]:
policy_cited_scopus1.head()

,eids,len,impact,cumsum_eid_len,i,i2,is_TPIS,country,is_rich_club,authid,name,main_country,main_aff,affliation
0,,,,,,,,,,,,,,
7003499456,"[84940239344, 84939985700, 85045740431, 849518...",70,14.138443,0.005637,0,0.000000,1,che,YES,7003499456,Seneviratne_Sonia I.,che,Institute for Atmospheric and Climate Science_che,Institute for Atmospheric and Climate Science
36660575800,"[84930377030, 84937073408, 84958059282, 849736...",28,9.806385,0.007792,1,0.000028,1,sau,YES,36660575800,Almazroui_Mansour,sau,King Abdulaziz University_sau,King Abdulaziz University
55399842300,"[84964047746, 85046023945, 85072783484, 850161...",95,7.918689,0.014589,2,0.000055,1,fra,YES,55399842300,Ciais_Philippe,fra,Université Paris-Saclay_fra,Université Paris-Saclay
57202301596,"[84988422799, 84992417412, 85010736859, 850295...",34,7.483705,0.017076,3,0.000083,1,usa,YES,57202301596,Xie_Shang-Ping,usa,University of California San Diego_usa,University of California San Diego
55388694300,"[84957434131, 85020496593, 85060570271, 850739...",29,7.423947,0.019397,4,0.000111,1,usa,YES,55388694300,Wehner_Michael F.,usa,Lawrence Berkeley National Laboratory_usa,Lawrence Berkeley National Laboratory


In [4]:
policy_cited_scopus1.shape

(36143, 14)

In [5]:
policy_cited_scopus2 = policy_cited_scopus1

In [6]:
import ast

policy_cited_scopus2['eids'] = policy_cited_scopus2['eids'].apply(ast.literal_eval)

In [7]:
policy_cited_scopus2.shape

(36143, 14)

In [8]:
policy_cited_scopus2.head()

,eids,len,impact,cumsum_eid_len,i,i2,is_TPIS,country,is_rich_club,authid,name,main_country,main_aff,affliation
0,,,,,,,,,,,,,,
7003499456,"[84940239344, 84939985700, 85045740431, 849518...",70,14.138443,0.005637,0,0.000000,1,che,YES,7003499456,Seneviratne_Sonia I.,che,Institute for Atmospheric and Climate Science_che,Institute for Atmospheric and Climate Science
36660575800,"[84930377030, 84937073408, 84958059282, 849736...",28,9.806385,0.007792,1,0.000028,1,sau,YES,36660575800,Almazroui_Mansour,sau,King Abdulaziz University_sau,King Abdulaziz University
55399842300,"[84964047746, 85046023945, 85072783484, 850161...",95,7.918689,0.014589,2,0.000055,1,fra,YES,55399842300,Ciais_Philippe,fra,Université Paris-Saclay_fra,Université Paris-Saclay
57202301596,"[84988422799, 84992417412, 85010736859, 850295...",34,7.483705,0.017076,3,0.000083,1,usa,YES,57202301596,Xie_Shang-Ping,usa,University of California San Diego_usa,University of California San Diego
55388694300,"[84957434131, 85020496593, 85060570271, 850739...",29,7.423947,0.019397,4,0.000111,1,usa,YES,55388694300,Wehner_Michael F.,usa,Lawrence Berkeley National Laboratory_usa,Lawrence Berkeley National Laboratory


In [9]:
target_eids = policy_cited_scopus2.explode(['eids'])['eids'].unique()

In [10]:
len(target_eids)

12064

In [11]:
target_eids_TPIS = policy_cited_scopus2.explode(['eids'])[['eids','is_TPIS']].reset_index()
target_eids_TPIS = target_eids_TPIS[['eids','is_TPIS']]
target_eids_TPIS.drop_duplicates('eids', inplace=True)
target_eids_TPIS.rename(columns={'eids':'eid'},inplace=True)

In [12]:
target_eids_TPIS.value_counts('is_TPIS')

is_TPIS
0    8443
1    3621
Name: count, dtype: int64

In [13]:
abstract_df = pd.read_pickle('/disks/qnap3/shared/scopus-24/data/paper/abstract.pickle')
abstract_df = pd.DataFrame(abstract_df)
abstract_df.reset_index(inplace=True)
abstract_df.rename(columns={'index':'eid'},inplace=True)

In [14]:
title_df = pd.read_pickle('/disks/qnap3/shared/scopus-24/data/paper/title.pickle')
title_df = pd.DataFrame(title_df)
title_df.reset_index(inplace=True)
title_df.rename(columns={'index':'eid'},inplace=True)

In [15]:
year_df = pd.read_pickle('/disks/qnap3/shared/scopus-24/data/paper/year.pickle')
year_df = pd.DataFrame(year_df)
year_df.reset_index(inplace=True)
year_df.rename(columns={'index':'eid'},inplace=True)

In [16]:
title_df.head()

,eid,title
0,107,Assessing surface-atmosphere interactions usin...
1,110,Symmetry of quantum phase space in a degenerat...
2,116,Friction in strongly confined polymer melts: E...
3,117,Nearby young dwarf galaxies: Primordial gas an...
4,118,Determination of iodate in salt samples with a...


In [17]:
abstract_df1 = abstract_df[abstract_df['eid'].isin(target_eids)]

In [18]:
title_df1 = title_df[title_df['eid'].isin(target_eids)]

In [19]:
policy_cited_scopus = pd.merge(title_df1, abstract_df1, on='eid', how='left')

In [20]:
policy_cited_scopus['text']= policy_cited_scopus['title'] + ' .'  + policy_cited_scopus['abstract']

In [21]:
policy_cited_scopus = pd.merge(policy_cited_scopus, year_df, on='eid', how='left')

In [22]:
policy_cited_scopus = pd.merge(policy_cited_scopus, target_eids_TPIS, on='eid', how='left')

In [23]:
policy_cited_scopus.head()

,eid,title,abstract,text,year,is_TPIS
0,84881489049,Effectiveness of Hydromulching to Reduce Runof...,Forest fires can greatly increase runoff and s...,Effectiveness of Hydromulching to Reduce Runof...,2016,0
1,84886820230,Can savanna burning projects deliver measurabl...,Savannas constitute the most fire-prone vegeta...,Can savanna burning projects deliver measurabl...,2017,0
2,84893867446,Assessing the capability of CORDEX models in s...,Reliable forecasts of rainfall-onset dates (RO...,Assessing the capability of CORDEX models in s...,2015,1
3,84896410024,Changes in the characteristics of precipitatio...,"Based on observed daily precipitation data, th...",Changes in the characteristics of precipitatio...,2015,0
4,84896421802,Projections of climate change impacts on flood...,"Under a warming climate, changes in hydrologic...",Projections of climate change impacts on flood...,2015,0


In [24]:
pd.options.display.max_columns = None
pd.options.display.max_rows = 100
pd.options.display.max_colwidth = None

In [25]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
import networkx as nx
from ast import literal_eval
import leidenalg as la
import community as community_louvain
import igraph as ig,leidenalg
import random



flatten2 = lambda l: [item for sublist in l if sublist == sublist for item in sublist]


def kamada(G):
    
    g = ig.Graph.TupleList(pd.DataFrame(G.edges()).itertuples(index=False), directed=False)
    
    return {v['name']:p for v,p in zip(g.vs(),g.layout_kamada_kawai()) }


In [28]:
import nltk
from nltk.corpus import stopwords
sw = set(stopwords.words('english'))
sw = sw | set(['<span','(<span', '>', "al.", "al.", '°c', '(last', 'idcombining', 'classcombining', 'mathvariantcombining', 'overflowcombining', 'under:', ')',
               'md5hashcombining', 'idcombining', 'heightcombining', 'xlink:hrefcombining', 'xmlnscombining',"(formula",
              "=","(n","presented.)","~","∼", "al.","el","2020","2021","2022","2019","2018","2017","2016","2015","significant","significantly"])

rm = {"al. 2019).", "et al. 2020).", "al. 2020).", "et al. 2019).", "al. 2019)", "1979–2018 period", "2019) al. et", "2019 january", "11 global","2019) al. et"}


def is_symble_(s):
    if 'line"' in s or "xmlns:" in s or "displaycombining" in s or "dspmathcombining" in s:
        return True
    if 'inline-formula' in s:
        return True

def not_symble(b):
    for s in b:
        if is_symble_(s):
            return False
    return True

not_symble(("a",'line","aaa"'))

False

In [31]:

    

def get_trigram(x):
    res = [ [" ".join(b).replace(",","") for b in  list(nltk.trigrams(t.lower().split())) if len(set(b) & set(sw))==0  and not_symble(b)    ]  for t in x.split(". ")]
    return flatten2(res)

def get_bigram(x):
    res = [ [" ".join(b).replace(",","") for b in  list(nltk.bigrams(t.lower().split())) if len(set(b) & set(sw))==0  and not_symble(b)    ]  for t in x.split(". ")]
    return flatten2(res)
    
policy_cited_scopus['keyword'] = policy_cited_scopus['text'].apply(lambda x: get_bigram(x) + get_trigram(x) ).apply(set).apply(lambda x: x-rm)

In [35]:
def remove_duplicates(phrases):
    return list(set(phrases))
from nltk.corpus import wordnet
# Function to find the base form (lemma) of a word using WordNet
def get_lemma(word):
    lemma = wordnet.morphy(word)
    return lemma if lemma else word

# Function to standardize phrases by converting all words to their base forms
def standardize_by_lemma(phrases):
    standardized_phrases = []
    for phrase in phrases:  # phrasesはすでにリスト形式
        words = phrase.lower().split()
        lemma_words = [get_lemma(word) for word in words]
        sorted_lemma_phrase = ' '.join(sorted(lemma_words))
        standardized_phrases.append(sorted_lemma_phrase)
    return standardized_phrases

# 'keyword' 列がリスト形式のデータに対して適用する
policy_cited_scopus['standardized_keyword'] = policy_cited_scopus['keyword'].apply(standardize_by_lemma)

# 重複を除去
policy_cited_scopus['standardized_keyword'] = policy_cited_scopus['standardized_keyword'].apply(remove_duplicates)

In [36]:
policy_cited_scopus_2018 = policy_cited_scopus[policy_cited_scopus['year']>=2018]

In [37]:
year_words_list = policy_cited_scopus.groupby('year')['standardized_keyword'].agg(flatten2)
year_words = year_words_list.apply(set)


In [38]:
word_count = pd.Series(collections.Counter(flatten2(year_words_list)))

In [39]:
word_count

date rainfall-onset                1
climate model                   1883
show wrf                           3
ccrm5 rca35 use                    1
remarkable skill                   2
                                ... 
.no territory                      1
2023 state                         1
.no territory uncharted            1
entering report:                   1
entering territory uncharted       1
Length: 888292, dtype: int64

In [40]:
target_words = set(word_count[word_count>=5].index)
len(target_words)

30379

In [41]:
total_word = set()
new_word ={}
for k,v in year_words.sort_index().items():
    new_word[k] =   (v - total_word) & target_words
    total_word = total_word | v
    print(len(total_word))

158927
308400
447956
595567
712510
811388
861373
883008
888292


In [42]:
pd.Series(new_word).apply(len)

2015    21969
2016     5844
2017     1855
2018      586
2019       93
2020       32
2021        0
2022        0
2023        0
dtype: int64

In [43]:
word_count_richclub = pd.concat([pd.Series(collections.Counter(flatten2(policy_cited_scopus.query('is_TPIS == 1').standardized_keyword) )),
           pd.Series(collections.Counter(flatten2(policy_cited_scopus.standardized_keyword) ))],axis=1).fillna(0)
word_count_richclub.columns = ['RC','total']
word_count_richclub['RC_ratio'] = word_count_richclub.RC / word_count_richclub.total

In [44]:
pd.set_option("display.max_rows", 300)

res = pd.Series(new_word).loc[2015:].explode().to_frame()
res['RC_ratio'] = res[0].map(word_count_richclub['RC_ratio'])
res = res.dropna()
res.rename(columns={0:'standardized_keyword'},inplace=True)
res = res.sort_values('RC_ratio')

In [45]:
# res.to_csv('for_gpt5.csv')

In [46]:
res = res[(res['standardized_keyword']!='2019) al. et') & (res['standardized_keyword']!='2019 january')]

In [47]:
len(res)

30377

In [48]:
# 論文の比率
policy_cited_scopus2018 = policy_cited_scopus[policy_cited_scopus['year']>=2015]
policy_cited_scopus2018['is_TPIS'].value_counts()/len(policy_cited_scopus2018),policy_cited_scopus2018['is_TPIS'].value_counts(), len(policy_cited_scopus2018)

(is_TPIS
 0    0.699851
 1    0.300149
 Name: count, dtype: float64,
 is_TPIS
 0    8443
 1    3621
 Name: count, dtype: int64,
 12064)

In [49]:
4679 / (4679 + 2162), 2162 / (4679 + 2162), (4679 + 2162)

(0.6839643326998976, 0.31603566730010235, 6841)

In [50]:
# Create a new column 'category' to classify based on 'RC_ratio'
# and reclassify 'standardized_keyword' into lists based on those categories

# Define the conditions for the classificationd
data = res

conditions = [
    (data['RC_ratio'] == 0),
    (data['RC_ratio'] > 0) & (data['RC_ratio'] < 0.3),
    (data['RC_ratio'] == 0.3),
    (data['RC_ratio'] > 0.3) & (data['RC_ratio'] < 1),
    (data['RC_ratio'] == 1)
]

# Define the corresponding category labels
category_labels = ['RC_ratio_0', 'RC_ratio_0_to_0312', 'RC_ratio_03', 'RC_ratio_03to_1', 'RC_ratio_1']

# Assign the category labels
data['category'] = pd.cut(data['RC_ratio'], bins=[-0.01, 0, 0.3, 0.999, 1], 
                          labels=['RC_ratio_0', 'RC_ratio_0_to_03', 'RC_ratio_03_to_1', 'RC_ratio_1'], include_lowest=True)

# Group the standardized keywords by their respective categories into lists
categorized_keywords = data.groupby('category')['standardized_keyword'].apply(list).reset_index()

categorized_keywords

/tmp/ipykernel_3989542/2943615204.py:23: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  categorized_keywords = data.groupby('category')['standardized_keyword'].apply(list).reset_index()


,category,standardized_keyword
0,RC_ratio_0,"[base plateau, investigate two, measurement salinity, lake within, bp) ka, productivity terrestrial, crop evapotranspiration, high pm, moderate show, region require, dry early season, product rainfall satellite, c tg, aspect slope, analysis precipitation, product satellite-based, cool season, particular reference, dust increase, acknowledge widely, chronology tree-ring width, fatality flood, altimeter radar, due quality, forest oak, southwestern usa, global ice sheet, provision service, (decreasing) increase, forecast global, locate watershed, black carbon emission, severe soil, evergreen shrub, distribution pattern spatial, different zone, air modeling quality, aura monitoring ozone, land terrestrial, flow natural, northern route sea, basin heihe river, level three, 3d model, human intensive, pine plantation, intensity vary, species threaten, angstrom exponent, represent source, maximum temperature trend, socioeconomic status, due flow, central mexico, change source, fitting spectral, better substantially, ha year, (fews net), meteorological simulation, lake model, (figure presented)., data reflectance, .natural gas, select study, analysis flow, among compare, estimation rainfall, column-averaged mole, management soil, per tons, basin outlet, lake using, empirical orthogonal using, blanc massif, level sea using, could exacerbate, co estimate flux, bark beetle spruce, information need, 1960–2010 period, flux model, management objective, account site, criterion selection, boreal manage, increase step, measure snow, lower moisture soil, enso response, matter organic terrestrial, change climate holocene, age exposure, due lost, iron solubility, bp cal yr, ambient conditions, .this change study, 1.8 ±, decade using, ...]"
1,RC_ratio_0_to_03,"[information system, china plain, gas production, aquatic ecosystem, emission rate, assessment tool water, china north plain, air model quality, monitoring system, forest soil, (swat) tool, (swat) assessment tool, gas natural, region western, area increase, food web, environmental factor, large second, .the data, holocene late, basin scale, loess plateau, column water, frequently occur, gas natural production, availability nutrient, soil type, assessment water, landsat using, higher rates, arctic ecosystem, air annual mean, salinity surface, earlier snowmelt, (doc) carbon, cover land type, (doc) carbon organic, area irrigate, carbon stable, environmental protection, (pbl) boundary layer, african south, lagrangian particle, (pbl) layer, conservation water, aral sea, crop water, atmospheric dust, forest tree, experiment laboratory, forest stand, forest growth, northwest territory, climate predict, agency meteorological, forest national, many years, aquifer coastal, emission nh, inorganic nitrogen, invasive species, global modeling, model quality, ground-level ozone, temperature time, average emission, agricultural productivity, effects interactive, intrusion seawater, flow groundwater, lake victoria, composition isotope, force wind, diversity genetic, content moisture soil, hierarchy process, change. global, river yellow, ecosystem important, present review, distribution species, driver potential, also paper, concentration measure, show that:, density tree, conductivity hydraulic, aerosol source, co observation, ecological restoration, dispersion model particle, inorganic secondary, dispersion particle, forest type, control process, community local, crop growth, action management, data exist, many species, ...]"
2,RC_ratio_03_to_1,"[result suggest, budget energy, 10 km, boreal winter, observation using, improve model, historical trend, center national, many study, length season, indian summer, minimum temperature, model produce, 2 km, analysis indicate, pattern teleconnection, major river, distribution vertical, pacific subtropical, dominant factor, future work, ozone precursor, ocean pacific, ice sea, ecosystem terrestrial, burn global, (tws) storage

## Analysis from 2018 onwards

In [51]:
# リストや配列の要素をすべて表示する
pd.set_option('display.max_seq_items', None)

# (オプション) 各列の幅（文字数）を広げて、長いリストが途中で切れないようにする
pd.set_option('display.max_colwidth', None)

In [52]:

res = pd.Series(new_word).loc[2018:].explode().to_frame()
res['RC_ratio'] = res[0].map(word_count_richclub['RC_ratio'])
res = res.dropna()
res.rename(columns={0:'standardized_keyword'},inplace=True)
res = res.sort_values('RC_ratio')
res = res[(res['standardized_keyword']!='2019) al. et') & (res['standardized_keyword']!='2019 january')& (res['standardized_keyword']!='2018) al.')]

# Define the conditions for the classificationd
data1 = res

conditions = [
    (data['RC_ratio'] == 0),
    (data['RC_ratio'] > 0) & (data['RC_ratio'] < 0.3),
    (data['RC_ratio'] == 0.3),
    (data['RC_ratio'] > 0.3) & (data['RC_ratio'] < 1),
    (data['RC_ratio'] == 1)
]

# Define the corresponding category labels
category_labels = ['PPR 0', 'PPR 0 to 0.3', 'PPR 0.3', 'RRP 0.3 to 1', 'PPR 1']

# Assign the category labels
data1['category'] = pd.cut(data1['RC_ratio'], bins=[-0.01, 0, 0.3, 0.999, 1], 
                          labels=['PPR 0', 'PPR 0 to 0.3', 'RRP 0.3 to 1', 'PPR 1'], include_lowest=True)

# Group the standardized keywords by their respective categories into lists
categorized_keywords1 = data1.groupby('category')['standardized_keyword'].apply(list).reset_index()

categorized_keywords1

/tmp/ipykernel_3989542/875041947.py:27: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  categorized_keywords1 = data1.groupby('category')['standardized_keyword'].apply(list).reset_index()


category  \
0         PPR 0   
1  PPR 0 to 0.3   
2  RRP 0.3 to 1   
3         PPR 1   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

In [53]:
policy_cited_scopus_2018 = policy_cited_scopus[policy_cited_scopus['year']>=2017]
year_words_list = policy_cited_scopus_2018.groupby('year')['standardized_keyword'].agg(flatten2)
word_count = pd.Series(collections.Counter(flatten2(year_words_list)))

# 'standardized_keyword' カラム（リストが格納されている）に apply を適用
categorized_keywords1['standardized_keyword'] = categorized_keywords1['standardized_keyword'].apply(
    lambda word_list: sorted(
        word_list, 
        # 各 word のキーとして word_count Series から登場回数を取得する
        # .get(word, 0) を使い、万が一 word_count に単語が存在しない場合は 0 を返す
        key=lambda word: word_count.get(word, 0), 
        reverse=True # 降順 (登場回数が多い順)
    )
)

# ソート結果の確認
categorized_keywords1
categorized_keywords1['top_15_keywords'] = categorized_keywords1['standardized_keyword'].apply(lambda x: x[:15])
categorized_keywords1

category  \
0         PPR 0   
1  PPR 0 to 0.3   
2  RRP 0.3 to 1   
3         PPR 1   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       